<a href="https://colab.research.google.com/github/suva1998/hsi-lithology-showcase/blob/main/notebooks/03_train_baseline_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03: Train Baseline CNN on Geological Drill Cores

This notebook trains the Baseline CNN on the **HZDR RODARE Upscaling Mineralogy Benchmark** (Thiele et al., 2026). It demonstrates how to handle hyperspectral drill-core data with class imbalances and multi-scale spatial features.

In [ ]:
# 1. Setup Environment (Colab)
!git clone https://github.com/suva1998/hsi-lithology-showcase.git
%cd hsi-lithology-showcase
!pip install -e .

In [ ]:
# Mount Google Drive to save model checkpoints persistently
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab. Checkpoints will be saved locally.")

In [ ]:
# 2. Load Configuration
import yaml
from pprint import pprint

with open('configs/mla_drillcore.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Training Configuration:")
pprint(config['training'])

In [ ]:
# 3. Download Data
from src.data import download

data, labels = download.download_dataset("mla_drillcore", config['paths']['data_dir'])

# 4. Preprocess Data

Before training, we must apply the pipeline from `02_preprocessing_pipeline.ipynb`. This normalizes the data, applies FastICA, extracts patches, and performs a stratified split.

In [ ]:
from src.data.preprocessing import preprocess_hsi
import time

print("Starting preprocessing pipeline... (This takes a moment due to FastICA)")
start_time = time.time()
result = preprocess_hsi(data, labels, config)
print(f"\nPreprocessing complete in {time.time() - start_time:.1f}s")

# 5. Build and Train Baseline CNN

Now we initialize the `Trainer` class. The trainer automatically:
1. Builds the Baseline CNN with dilated separable convolutions.
2. Configures the class-balanced Focal Loss to handle rare minerals.
3. Applies MixUp augmentation to the training patches.
4. Executes the training loop with Cosine Decay learning rate scheduling.

In [ ]:
from src.training.trainer import Trainer

# Initialize the trainer with the YAML config
trainer = Trainer(config)

# Train the model on the generated splits
# (This will train for the number of epochs specified in configs/mla_drillcore.yaml)
training_result = trainer.train(
    splits=result['splits'],
    class_counts=result['class_counts']
)

# 6. Evaluate and Visualize Training History

Let's plot the training vs validation accuracy and loss to ensure our model converged properly and didn't overfit.

In [ ]:
import matplotlib.pyplot as plt

history = training_result['history']
epochs = range(1, len(history['loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot Loss
ax1.plot(epochs, history['loss'], 'b-', label='Training Loss')
ax1.plot(epochs, history['val_loss'], 'r-', label='Validation Loss')
ax1.set_title('Class-Balanced Focal Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.6)

# Plot Accuracy (FIXED from ax1 to ax2)
ax2.plot(epochs, history['accuracy'], 'b-', label='Training Accuracy')
ax2.plot(epochs, history['val_accuracy'], 'r-', label='Validation Accuracy')
ax2.set_title('Classification Accuracy')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import numpy as np

# 1. Get the completely unseen Test splits
X_test, y_test = result['splits']['test']

# 2. Ask the model to predict the test set
print("Predicting on Test Set...")
y_pred_probs = training_result['model'].predict(X_test, batch_size=64)
y_pred = np.argmax(y_pred_probs, axis=-1)

# 3. Generate Classification Report (Precision, Recall, F1-Score)
target_names = [class_names[i] for i in np.unique(y_test)]
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# 4. Plot Confusion Matrix
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=target_names,
    cmap='Blues',
    ax=ax
)
plt.title("Test Set Confusion Matrix")
plt.show()


In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import glob
from src.data.preprocessing import prepare_rgb_patches

# 1. Select a spatial chunk (e.g., the top 1000 rows) for visualization
raw_cube = data[:1000, :, :]
raw_labels = labels[:1000, :]
h, w, b = raw_cube.shape

print("Applying ICA dimensionality reduction to the visualization core...")
scaler = result['scaler']
reducer = result['reducer']

# Apply the identical normalization and FastICA to the raw core
pixels = raw_cube.reshape(-1, b)
pixels_norm = scaler.transform(pixels)
pixels_reduced = reducer.transform(pixels_norm)
reduced_cube = pixels_reduced.reshape(h, w, -1)

# 2. Extract coordinates and patches for all valid foreground pixels
valid_y, valid_x = np.nonzero(raw_labels > 0)
print(f"Predicting {len(valid_y)} pixels for the visualization core...")

patch_size = config['dataset']['patch_size']
margin = patch_size // 2
padded_cube = np.pad(reduced_cube, ((margin, margin), (margin, margin), (0, 0)), mode="constant")

# Extract the test patches
patches = []
for y, x in zip(valid_y, valid_x):
    patches.append(padded_cube[y:y + patch_size, x:x + patch_size, :])
patches = np.array(patches)

print("Formatting patches for Baseline CNN...")
patches_rgb = prepare_rgb_patches(patches)

# 3. Predict the labels across the entire rock surface!
print("Running CNN spatial inference...")
y_pred_probs = training_result['model'].predict(patches_rgb, batch_size=128, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=-1)

# 4. Reconstruct the 2D spatial prediction map
prediction_map = np.zeros((h, w), dtype=np.int32)
prediction_map[valid_y, valid_x] = y_pred

# Create a False-Color RGB composite for the raw rock (using bands 10, 20, 30)
rgb = np.zeros((h, w, 3))
for i, band in enumerate([10, 20, 30]):
    band_data = raw_cube[:, :, band]
    p_low, p_high = np.percentile(band_data, [2, 98])
    rgb[..., i] = np.clip((band_data - p_low) / (p_high - p_low + 1e-8), 0, 1)

# 5. Plot the three images side-by-side
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 8))

ax1.imshow(rgb)
ax1.set_title("RGB Composite (False Color)")
ax1.axis('off')

# Use a distinct colormap
cmap = plt.cm.get_cmap('tab20', 21)

ax2.imshow(raw_labels, cmap=cmap, interpolation='none', vmin=0, vmax=20)
ax2.set_title("Ground Truth Mineralogy (MLA)")
ax2.axis('off')

ax3.imshow(prediction_map, cmap=cmap, interpolation='none', vmin=0, vmax=20)
ax3.set_title("Baseline CNN Prediction")
ax3.axis('off')

# --- EXTRACT MINERAL NAMES FOR THE LEGEND ---
excel_path = glob.glob(f"{config['paths']['data_dir']}/**/AbundanceMapping.xlsx", recursive=True)[0]
df = pd.read_excel(excel_path)
macro_groups = df['Assigned mineral group'].dropna().unique().tolist()
class_names = ["Background"] + macro_groups

# Create legend only for minerals actually present in this core
present_classes = np.unique(np.concatenate([raw_labels.flatten(), prediction_map.flatten()]))
legend_patches = [mpatches.Patch(color=cmap(i), label=class_names[i]) for i in present_classes]


# Add the legend outside the plot
fig.legend(handles=legend_patches, loc='center right', title="Mineralogy", bbox_to_anchor=(1.10, 0.5), fontsize=12)

# Adjust layout to make room for legend
plt.tight_layout(rect=[0, 0, 0.95, 1])

# Save it to the results directory!
os.makedirs('experiments/results', exist_ok=True)
save_path = 'experiments/results/spatial_prediction_core0.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Masterpiece saved to: {save_path}")


In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from src.data.preprocessing import extract_patches, prepare_rgb_patches

print("Analyzing all drill cores and generating Excel Summary...")

patch_size = config['dataset']['patch_size']

# 1. Transform the full dataset exactly like during preprocessing
h, w, b = data.shape
pixels = data.reshape(-1, b)
pixels_norm = result['scaler'].transform(pixels)
pixels_reduced = result['reducer'].transform(pixels_norm)

# Cast to float32 to exactly match the training data types
data_reduced = pixels_reduced.reshape(h, w, -1).astype(np.float32)

# 2. Extract ALL valid patches using the exact same function from training
patches, patch_labels = extract_patches(data_reduced, labels, patch_size=patch_size)

# 3. Format for Baseline CNN and Predict EVERYTHING in one batch
patches_rgb = prepare_rgb_patches(patches)
y_pred_probs = training_result['model'].predict(patches_rgb, batch_size=256, verbose=0)
y_preds = np.argmax(y_pred_probs, axis=-1)

# --- SANITY CHECK ---
accuracy = (y_preds == patch_labels).mean()
print(f"Sanity Check - Overall Accuracy on Full Dataset: {accuracy * 100:.2f}%")
# --------------------

# 4. Now loop through 50-pixel "chunks" just to generate the Excel summary rows
results = []
chunk_size = 50
num_chunks = int(np.ceil(h / chunk_size))

# We need the valid coordinates to know which patch belongs to which chunk
valid_y, valid_x = np.nonzero(labels > 0)

# Extract real mineral names for the legend
excel_path = glob.glob(f"{config['paths']['data_dir']}/**/AbundanceMapping.xlsx", recursive=True)[0]
df = pd.read_excel(excel_path)
macro_groups = df['Assigned mineral group'].dropna().unique().tolist()
class_names = ["Background"] + macro_groups

for core_idx in range(num_chunks):
    start_y = core_idx * chunk_size
    end_y = min((core_idx + 1) * chunk_size, h)

    # Find which of our valid predictions belong to this physical chunk
    chunk_mask = (valid_y >= start_y) & (valid_y < end_y)

    if not np.any(chunk_mask):
        continue # Skip chunks that are purely background

    chunk_true_labels = patch_labels[chunk_mask]
    chunk_preds = y_preds[chunk_mask]
    chunk_pred_probs = y_pred_probs[chunk_mask]

    # Calculate dominant minerals
    true_counts = np.bincount(chunk_true_labels)
    pred_counts = np.bincount(chunk_preds)

    dominant_true_idx = np.argmax(true_counts)
    dominant_pred_idx = np.argmax(pred_counts)

    # Calculate confidence and accuracy
    dominant_confidence = np.mean(chunk_pred_probs[chunk_preds == dominant_pred_idx, dominant_pred_idx]) if len(chunk_pred_probs[chunk_preds == dominant_pred_idx]) > 0 else 0
    core_accuracy = (chunk_preds == chunk_true_labels).mean()

    results.append({
        "Drillcore_ID": f"Stonepark_Core_{core_idx:03d}",
        "Spatial_Dimensions": f"{end_y - start_y}x{w}",
        "Valid_Rock_Pixels": int(np.sum(chunk_mask)),
        "Dominant_True_Mineral": class_names[dominant_true_idx] if dominant_true_idx < len(class_names) else "Unknown",
        "Dominant_Predicted_Mineral": class_names[dominant_pred_idx] if dominant_pred_idx < len(class_names) else "Unknown",
        "Prediction_Confidence": f"{dominant_confidence * 100:.1f}%",
        "Overall_Accuracy": f"{core_accuracy * 100:.1f}%"
    })

# Convert to Pandas DataFrame
df_results = pd.DataFrame(results)

# Ensure directory exists and save to Excel
os.makedirs('experiments/results', exist_ok=True)
excel_path = 'experiments/results/drillcore_predictions_summary.xlsx'
df_results.to_excel(excel_path, index=False)

print(f"✅ Successfully saved Excel Sheet to: {excel_path}")
df_results.head(10)
